# Which map is random?

Reproduces the interactive figure from **Statistics Intuitions #1**.

Two patterns of 120 points each:

- **Poisson** — every point placed independently and uniformly.
- **Hard-core** — same, but rejecting any point closer than `MIN_SEP` to an existing one.

Then it runs the test R. D. Clarke published in one page in 1946 to show that V-1 impacts on
London were Poisson rather than clustered: divide the region into equal cells, count points per
cell, and compare the spread of those counts to what chance predicts.

Clarke, *An Application of the Poisson Distribution*, Journal of the Institute of Actuaries **72**
(1946), p.481.

Only dependency is `matplotlib`, and only for the final plot.

## The random number generator

The published figure runs in a browser, so to regenerate **the exact same patterns** this notebook
ports the same PRNG (mulberry32) rather than using Python's `random`. The assertion below checks
it against values produced by the JavaScript original.

In [ ]:
MASK = 0xFFFFFFFF

def _imul(a, b):
    """Low 32 bits of a 32-bit multiply — Python's stand-in for JS Math.imul."""
    return (a * b) & MASK

def mulberry32(seed):
    """Port of the mulberry32 PRNG used by the published figure."""
    a = seed & MASK
    def rnd():
        nonlocal a
        a = (a + 0x6D2B79F5) & MASK
        t = _imul(a ^ (a >> 15), 1 | a)
        t = ((t + _imul(t ^ (t >> 7), 61 | t)) & MASK) ^ t
        return ((t ^ (t >> 14)) & MASK) / 4294967296
    return rnd

# Values emitted by the JavaScript implementation for this seed.
_probe = mulberry32(20260817 ^ 0x85EBCA6B)
assert [round(_probe(), 12) for _ in range(3)] == [0.631401259918, 0.253607654944, 0.679552925983]
print("PRNG matches the browser implementation")

## The two point processes

In [ ]:
N        = 120        # points in each square
MIN_SEP  = 0.028      # hard-core minimum separation, in units of the square's side
SEED     = 20260817   # the seed shown under the published figure

def poisson_points(rng, n):
    """Every point independent and uniform. This is the truly random one."""
    return [(rng(), rng()) for _ in range(n)]

def hardcore_points(rng, n, min_sep):
    """Uniform points, rejecting any that land within min_sep of an existing point."""
    pts, r2, attempts = [], min_sep ** 2, 0
    while len(pts) < n and attempts < n * 8000:
        attempts += 1
        x, y = rng(), rng()
        if all((px - x) ** 2 + (py - y) ** 2 >= r2 for px, py in pts):
            pts.append((x, y))
    if len(pts) < n:
        raise RuntimeError(f"only placed {len(pts)} of {n} points")
    return pts

# The figure derives an independent stream for each pattern from the displayed seed.
random_pattern    = poisson_points(mulberry32(SEED ^ 0x9E3779B9), N)
constrained_pattern = hardcore_points(mulberry32(SEED ^ 0x85EBCA6B), N, MIN_SEP)
print(len(random_pattern), len(constrained_pattern))

## Clarke's test

Split the square into a `K x K` grid, count the points in every cell, and look at how often cells
hold 0, 1, 2, ... points.

The number to watch is the **variance-to-mean ratio** of those counts. Scatter `N` points
independently over `M` cells and each cell's count is Binomial(`N`, `1/M`), so

    mean     = N/M = lambda
    variance = lambda * (1 - 1/M)
    ratio    = 1 - 1/M          (0.99 for M = 100)

Anything more evenly spread than chance comes out below that; anything clustered comes out above.

In [ ]:
K = 10                      # 10 x 10 grid, so lambda = 120/100 = 1.2 (Clarke's was 0.93)
MAX_BIN = 4                 # last histogram bin is "4 or more"

def cell_counts(pts, k):
    counts = [0] * (k * k)
    for x, y in pts:
        i = min(k - 1, int(x * k))
        j = min(k - 1, int(y * k))
        counts[j * k + i] += 1
    return counts

def count_histogram(counts, max_bin):
    h = [0] * (max_bin + 1)
    for c in counts:
        h[min(c, max_bin)] += 1
    return h

def variance_mean_ratio(counts):
    m = sum(counts) / len(counts)
    v = sum((c - m) ** 2 for c in counts) / len(counts)
    return v / m

def poisson_expected(lam, n_cells, max_bin):
    from math import exp
    out, term, cumulative = [], exp(-lam), 0.0
    for r in range(max_bin + 1):
        if r == max_bin:
            out.append(n_cells * max(0.0, 1 - cumulative))
        else:
            out.append(n_cells * term)
            cumulative += term
            term = term * lam / (r + 1)
    return out

n_cells  = K * K
lam      = N / n_cells
theory   = 1 - 1 / n_cells
expected = poisson_expected(lam, n_cells, MAX_BIN)

for name, pts in [("random", random_pattern), ("constrained", constrained_pattern)]:
    counts = cell_counts(pts, K)
    print(f"{name:12s} histogram={count_histogram(counts, MAX_BIN)}  "
          f"variance/mean={variance_mean_ratio(counts):.2f}")
print(f"{'chance':12s} {'':30s}variance/mean={theory:.2f}")

## The figure

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(9, 9))
bins = list(range(MAX_BIN + 1))
labels = [str(b) for b in bins[:-1]] + [f"{bins[-1]}+"]

for col, (name, pts) in enumerate([("A - minimum spacing enforced", constrained_pattern),
                                   ("B - truly random", random_pattern)]):
    ax = axes[0][col]
    ax.scatter([p[0] for p in pts], [p[1] for p in pts], s=10, c="#1e293b")
    for g in range(K + 1):
        ax.axhline(g / K, lw=0.4, c="#2563eb", alpha=0.3)
        ax.axvline(g / K, lw=0.4, c="#2563eb", alpha=0.3)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(name, fontsize=10)

    counts = cell_counts(pts, K)
    ax = axes[1][col]
    ax.bar(bins, count_histogram(counts, MAX_BIN),
           color="#2563eb" if col else "#94a3b8", width=0.62)
    ax.step([b - 0.5 for b in bins] + [bins[-1] + 0.5],
            expected + [expected[-1]], where="post", color="#0f172a", ls="--", lw=1.5)
    ax.set_xticks(bins); ax.set_xticklabels(labels)
    ax.set_xlabel("points in a cell"); ax.set_ylabel("number of cells")
    ax.set_title(f"variance / mean = {variance_mean_ratio(counts):.2f}", fontsize=10)

fig.suptitle(f"Chance predicts a ratio of {theory:.2f}  (dashed line = Poisson expectation)")
fig.tight_layout()
plt.show()

## What to look for

Both squares hold the same number of points, so both histograms have the same mean. What differs
is the spread.

The truly random pattern lands near the dashed line: it has **more empty cells and more crowded
cells** than the pattern that looks random. Forbidding close pairs removes the crowded cells, and
removing those removes the empty ones too, pulling the histogram in from both ends and dropping
the ratio well below chance.

That is the whole illusion. Evenness is a sign of a constraint, not of chance.

Change `MIN_SEP` and re-run: as it falls toward zero the constrained pattern's ratio climbs back
toward the chance value, and the two squares become genuinely hard to tell apart.